# ACME Talent Insights

Analysis notebook for the ACME Talent ETL pipeline. It reads the **Gold** tables (and Silver, for the lookups) straight from the lake (MinIO, S3 API), with the same `Settings`/`Lake` code as `job.py`.

**Run it** after a successful pipeline run. Connection settings come from the environment: `LAKE_BUCKET`, `S3_ENDPOINT`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`. The `reports` container (`scripts/build_site.sh`) sets them and exports this notebook to `site/insights.html`.

The data is synthetic. Charts are aggregates; the two lookup helpers show ids and HR fields only, never names or emails.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import job
import findings
from build_dashboard import apply_style, LIGHT, SERIES

%config InlineBackend.figure_formats = ["svg"]
apply_style()
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"})

settings = job.Settings.from_env({**os.environ, "STEPS": "gold"})
lake = job.Lake.from_settings(settings)
lineage = lake.read_json("lineage/latest.json")
gold = job.read_gold(lake)
silver = job.read_silver(lake)
f = findings.compute_findings(gold)
checks = f"{sum(c['passed'] for c in lineage['reconciliation'])}/{len(lineage['reconciliation'])}"
print("lake:", settings.lake_bucket, "| run:", lineage["run_id"], lineage["status"], "| checks:", checks)
pd.Series({k: len(v) for k, v in gold.items()}, name="rows").to_frame()

In [ ]:
def label_bars(ax, bars, fmt="{:,.0f}", size=8):
    for b in bars:
        ax.annotate(fmt.format(b.get_height()), (b.get_x() + b.get_width() / 2, b.get_height()),
                    ha="center", va="bottom", fontsize=size, color=LIGHT["ink2"])


def hlabels(ax, bars, fmt="{:,.0f}", size=8):
    for b in bars:
        ax.annotate(" " + fmt.format(b.get_width()), (b.get_width(), b.get_y() + b.get_height() / 2),
                    va="center", fontsize=size, color=LIGHT["ink2"])

## 1. Pipeline: rows per dataset (Bronze vs Silver)
Bronze is a byte copy of landing; Silver keeps what passes validation. Only `learning_activity` loses rows (the planted bad lines are quarantined). Bronze counts JSONL *files*, so the event line count read by Silver is shown instead.

In [ ]:
steps = pd.DataFrame(lineage["steps"])
bronze = steps[steps.layer == "bronze"].set_index("dataset")["rows_out"]
silver_steps = steps[steps.layer == "silver"].set_index("dataset")
bronze.loc["learning_activity"] = silver_steps.loc["learning_activity", "rows_in"]  # event lines, not files
rows = pd.DataFrame({"Bronze": bronze, "Silver": silver_steps["rows_out"]}).loc[list(job.TABLES) + ["learning_activity"]]

ax = rows.plot.barh(color=SERIES[:2], width=0.75, figsize=(7.5, 4.2))
ax.invert_yaxis(); ax.set_xscale("log"); ax.set_xlabel("rows (log scale)"); ax.grid(axis="y", visible=False)
ax.set_title("Rows per dataset: Bronze vs Silver")
plt.show()
rows.assign(Quarantined=silver_steps["rows_rejected"])

## 2. Q1: Rating distribution by period

In [ ]:
hist = gold["gold_employee_review_history"]
dist = pd.crosstab(hist["rating"], hist["review_period"])
ax = dist.plot.bar(color=SERIES[:2], width=0.75, rot=0)
ax.set_title("Rating distribution by review period"); ax.set_ylabel("reviews"); ax.grid(axis="x", visible=False)
for c in ax.containers:
    label_bars(ax, c)
plt.show()
print("avg:", f["ratings"]["avg_by_period"], "| improved / same / declined %:",
      f["ratings"]["pct_improved"], f["ratings"]["pct_same"], f["ratings"]["pct_declined"])

## 3. Q1: Average rating by department and period

In [ ]:
latest = hist["review_period"].max()
dept_period = hist.pivot_table(index="department", columns="review_period", values="rating", aggfunc="mean").sort_values(latest)
ax = dept_period.plot.barh(color=SERIES[:2], width=0.75, figsize=(7.5, 4.6))
ax.set_xlim(2.5, 3.8); ax.set_xlabel("average rating (axis starts at 2.5)"); ax.grid(axis="y", visible=False)
ax.set_title("Average rating by department and period")
plt.show()
dept_period.round(2)

## 4. Q2: Top 15 competencies by critical gaps

In [ ]:
gaps = gold["gold_skill_gaps"]
top15 = gaps[gaps.gap_severity == "critical"].competency_name.value_counts().head(15).sort_values()
fig, ax = plt.subplots(figsize=(7.5, 4.6))
hlabels(ax, ax.barh(top15.index, top15.values, color=LIGHT["s1"]))
ax.set_title(f"Top 15 competencies by critical gaps ({f['skill_gaps']['critical']} critical of {len(gaps):,})")
ax.set_xlabel("critical gaps"); ax.grid(axis="y", visible=False)
plt.show()

## 5. Q3: Potential vs promotion readiness

In [ ]:
promo = gold["gold_promotion_readiness"]
ct = pd.crosstab(promo["potential_rating"], promo["promotion_readiness"]).reindex(
    index=["High", "Medium", "Low"], columns=["Ready Now", "1-2 Years", "Not Ready"])
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.imshow(ct.values, cmap="Blues")
ax.set_xticks(range(3), ct.columns); ax.set_yticks(range(3), ct.index); ax.grid(False)
for i in range(3):
    for j in range(3):
        v = ct.values[i, j]
        ax.text(j, i, f"{v}", ha="center", va="center", color="white" if v > ct.values.max() * 0.6 else LIGHT["ink"])
ax.set_xlabel("promotion readiness"); ax.set_ylabel("potential")
ax.set_title("Employees by potential and readiness")
plt.show()
print(f["promotion"])

## 6. Q3: Promotion-ready by job level
Promotion-ready = High potential + Ready Now + latest rating >= 4.

In [ ]:
level_order = ["IC1", "IC2", "IC3", "Senior IC", "Manager", "Senior Manager", "Director"]
by_level = promo.groupby("job_level")["is_promotion_ready"].sum().reindex(level_order)
fig, ax = plt.subplots()
label_bars(ax, ax.bar(by_level.index, by_level.values, color=LIGHT["s1"], width=0.6))
ax.set_title("Promotion-ready employees by job level"); ax.set_ylabel("employees"); ax.grid(axis="x", visible=False)
ax.tick_params(axis="x", labelrotation=25)
plt.show()

## 7. Q4: Learning events by type

In [ ]:
train = gold["gold_training_summary"]
ev = pd.Series({t: train[f"n_{t}"].sum() for t in job.EVENT_TYPES}).sort_values()
fig, ax = plt.subplots()
hlabels(ax, ax.barh(ev.index.str.replace("_", " "), ev.values, color=LIGHT["s1"]))
ax.set_title(f"Learning events by type ({ev.sum():,} total)"); ax.set_xlabel("events"); ax.grid(axis="y", visible=False)
plt.show()

## 8. Q4: Courses completed per employee

In [ ]:
cc = train["courses_completed"].value_counts().sort_index()
fig, ax = plt.subplots()
label_bars(ax, ax.bar(cc.index.astype(str), cc.values, color=LIGHT["s1"], width=0.7))
ax.set_title(f"Distinct courses completed per employee (mean {train.courses_completed.mean():.2f}; "
             f"{f['training']['no_completed_course']} with none)")
ax.set_xlabel("courses completed"); ax.set_ylabel("employees"); ax.grid(axis="x", visible=False)
plt.show()

## 9. Q5: Computed attrition risk tiers vs the HR flag

In [ ]:
attr = gold["gold_attrition_risk"]
tiers = ["Low", "Medium", "High"]
cmp_ = pd.DataFrame({"Computed tier": attr.computed_risk_tier.value_counts(),
                     "HR flag": attr.attrition_risk.value_counts()}).reindex(tiers)
ax = cmp_.plot.bar(color=SERIES[:2], width=0.7, rot=0)
for c in ax.containers:
    label_bars(ax, c)
ax.set_title(f"Risk tiers: computed score vs HR flag (as of {f['attrition']['as_of_date']})")
ax.set_ylabel("employees"); ax.grid(axis="x", visible=False)
plt.show()
print("high risk AND high potential:", f["attrition"]["high_risk_and_high_potential"])
pd.crosstab(attr.computed_risk_tier, attr.attrition_risk, margins=True).reindex(index=tiers + ["All"], columns=tiers + ["All"])

## 10. Q6: Team rating vs project outcome

In [ ]:
teams = gold["gold_team_performance"]
tp = teams[teams.has_projects]
slope = tp.avg_rating.cov(tp.avg_outcome_score) / tp.avg_outcome_score.var()
intercept = tp.avg_rating.mean() - slope * tp.avg_outcome_score.mean()
xs = pd.Series([tp.avg_outcome_score.min(), tp.avg_outcome_score.max()])
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(tp.avg_outcome_score, tp.avg_rating, s=40, color=LIGHT["s1"], edgecolor="white", linewidth=1, zorder=3)
ax.plot(xs, intercept + slope * xs, color=LIGHT["muted"], linestyle="--", linewidth=1.5)
ax.set_xlabel("average project outcome score"); ax.set_ylabel("team avg rating (latest period)")
ax.set_title(f"Team rating vs project outcome: r = {f['teams']['pearson_r']} (n = {len(tp)} teams)")
plt.show()
print("avg rating by on-time band:", f["teams"]["avg_rating_by_on_time_band"], "| teams:", f["teams"]["teams_by_on_time_band"])

## 11. Q7: Proficiency heatmap, department x category
Weighted by number of assessments.

In [ ]:
sd = gold["gold_skill_distribution"]
heat = findings.weighted(sd, ["department", "category"], "avg_proficiency").unstack("category")
fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(heat.values, cmap="Blues", vmin=heat.values.min() - 0.1, vmax=heat.values.max())
ax.set_xticks(range(heat.shape[1]), heat.columns); ax.set_yticks(range(heat.shape[0]), heat.index); ax.grid(False)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                color="white" if v > heat.values.mean() + 0.1 else LIGHT["ink"])
fig.colorbar(im, ax=ax, shrink=0.8, label="avg proficiency")
ax.set_title("Average proficiency by department and category")
plt.show()
print("strongest:", f["skills"]["strongest"], "| weakest:", f["skills"]["weakest"])

## 12. Q7: 15 lowest % meeting the required level

In [ ]:
by_comp = findings.weighted(sd, ["competency_name"], "pct_meeting_required").sort_values().head(15)[::-1]
overall = f["skills"]["overall_pct_meeting"]
fig, ax = plt.subplots(figsize=(7.5, 4.6))
hlabels(ax, ax.barh(by_comp.index, by_comp.values, color=LIGHT["s1"]), "{:.1f}%")
ax.axvline(overall, color=LIGHT["muted"], linestyle="--", linewidth=1)
ax.text(overall, len(by_comp) - 0.5, f" overall {overall:.0f}%", color=LIGHT["ink2"], fontsize=9, va="bottom")
ax.set_xlim(0, 100); ax.set_xlabel("% of assessments at or above required level"); ax.grid(axis="y", visible=False)
ax.set_title("15 competencies with the lowest share meeting the required level")
plt.show()

## Lookups
Per-employee views for analysts. They return ids and HR fields only (no names or emails), so the exported notebook stays safe to publish.

In [ ]:
def review_history(employee_id: int) -> pd.DataFrame:
    cols = ["employee_id", "department", "job_level", "review_period", "rating", "prev_rating",
            "rating_change", "submitted_date", "dq_before_hire"]
    return hist.loc[hist.employee_id == employee_id, cols].reset_index(drop=True)


def training_history(employee_id: int) -> pd.DataFrame:
    ev = silver["learning_activity"]
    cols = ["event_date", "event_type", "course_name", "score", "schema_drift"]
    return ev.loc[ev.employee_id == employee_id, cols].sort_values("event_date").reset_index(drop=True)


example = int(attr.loc[attr.computed_risk_tier == "High", "employee_id"].min())
print("example: lowest employee_id in the computed High risk tier ->", example)
display(review_history(example))
display(training_history(example).tail(10))

## Assumptions and limitations

- **Synthetic data.** Everything comes from `scripts/generate_data.py`; the numbers illustrate the method, not a real workforce.
- **Two review periods only** (2024-H2, 2025-H1). "Trend" means one period-over-period change, so `declining_rating` is a one-step signal.
- **The attrition score is rule-based, not a model.** `risk_score = 2*HR flag High + HR flag Medium + declining rating + latest rating <= 2 + no learning in 180 days`. It is explainable but unvalidated: there are no attrition outcomes to test it against, and the HR flag dominates by design.
- **"Recent" learning is measured from the newest event**, not today, so reruns give the same answer. If the feed stops, nobody becomes "inactive" by the clock alone.
- **Pre-hire records are kept and flagged** (reviews and assessments dated before `hire_date`). Dropping them would bias the averages; in real data they are most likely date-entry errors.
- **Quarantined events are excluded** (duplicates, missing employee ids, bad dates, malformed JSON), so training counts are slightly understated.
- **Team vs project outcome covers only the teams with projects.** That is a small sample, and correlation is not causation.
- **Skill gaps** use `required_level - proficiency_level`; "critical" means a gap of 2 or more. Each assessment is a single snapshot (no time series).
- **Department figures are weighted by number of assessments**, so departments with more assessments count more in the "overall" numbers.